In [ ]:
from active_learning_related.modules.smalltext_pipeline import SmallTextPipeline
from active_learning_related.modules.smalltext_pipeline import smalltext_config
import pickle
from small_text import(
    PredictionEntropy,
    DeltaFScore,
    ClassBalancer,
    RandomSampling
)
import pandas as pd
import numpy as np

In [ ]:
schizophrenia_pipeline = SmallTextPipeline.load_learning_state(
    ""
)

In [ ]:
with open('../data/small_text_datasets.pkl', 'rb') as f:
    small_text_datasets = pickle.load(f)

train_dataset = small_text_datasets['psyB']['smalltext_train_dset']
test_dataset = small_text_datasets['psyB']['smalltext_test_dset']

In [34]:
base_query_strategy = RandomSampling()
trainer_kwargs = {
    'num_epochs': 1,
    'body_learning_rate': 2e-5,
    'seed':42,
    'max_length': 128,
    'num_iterations': 20
}

clf_factory, clf_classifier = smalltext_config(
    train_data = train_dataset,
    num_classes=2,
    model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    strategy = base_query_strategy,
    subsampling=True,
    subsampling_size=100000,
    classification_kwargs = {
        'device': 'cuda',
        'mini_batch_size': 16
    },
    model_args = trainer_kwargs
)

In [35]:
schizophrenia_pipeline = SmallTextPipeline(
    train = train_dataset,
    test = test_dataset,
    disease = 'schizophrenia',
    clf_factory=clf_factory,
    active_learner=clf_classifier
)

In [ ]:
schizophrenia_pipeline.initialize_learner(indices_initial=range(0,50))

In [ ]:
schizophrenia_pipeline.start_loop_async(
    num_samples=100, 
    path_to_labels="", 
    path_to_disk=""
)

In [ ]:
schizophrenia_pipeline.continue_loop_async(
    path_to_labels = "", 
    path_to_disk=""
)

In [ ]:
schizophrenia_pipeline.save_learning_state(
    ""
)